# 🧠 Blueprint: How We Will Build Our Own LLM

This full course will follow the real pretraining pipeline used in modern models like Gemma, LLaMA, and GPT-style architectures.

---

## **1. Data Preparation**
- Download TinyStories
- Clean text
- Train/validation split
- Save clean text

---

## **2. Tokenization**
- Use GPT2 BPE tokenizer
- Convert text → token IDs
- Save as memory-efficient `.bin` files

---

## **3. Creating Input–Output Pairs**
- Define block size (context length)
- Create next-token prediction labels

---

## **4. Model Architecture (Gemma-Style)**
- Token embeddings (dim=640)
- 18 Transformer blocks
- RMSNorm everywhere
- Multi-Query Attention (MQA)
- Sliding Window Attention (512 window)
- RoPE
- GLU-based FFN

---

## **5. Training the Model**
- AdamW
- Warmup + cosine decay learning rate
- Mixed precision (FP16)
- Gradient accumulation
- Checkpointing

---

## **6. Inference & Deployment**
- Autoregressive text generation
- Temperature, top-k, top-p sampling
- Save/load weights
- Upload to Hugging Face Hub

---

## 🎯 Final Goal
**A fully functional 270M parameter Small Language Model that YOU built end-to-end.**


## 1. Check Python Environment + GPU

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — training will be extremely slow.")


PyTorch version: 2.9.0+cpu
CUDA available: False
No GPU detected — training will be extremely slow.


## 2. Install Required Dependencies

In [ ]:
!pip install datasets tiktoken numpy tqdm matplotlib einops accelerate


## 3. Create Project Folder Structure

In [ ]:
import os

base_path = "/content/drive/MyDrive/Courses/GenAI/SLM from Scratch/"


In [ ]:
folders = [
    "data", "data/raw", "data/tokenized",
    "model", "model/checkpoints", "model/logs", "model/config",
    "notebooks_output"
]

for f in folders:
    os.makedirs(base_path+f, exist_ok=True)

folders


['data', 'data/raw', 'data/tokenized', 'model', 'model/checkpoints', 'model/logs', 'model/config', 'notebooks_output']

## 4. Load a Preview of the TinyStories Dataset

In [ ]:
from datasets import load_dataset

print("Loading TinyStories preview...")
ds_stream = load_dataset("roneneldan/TinyStories", split="train", streaming=True)

preview = []
for i, sample in enumerate(ds_stream):
    preview.append(sample["text"])
    if i >= 4:
        break

preview


# DATA PREPARATION

In [ ]:
import os
from datasets import load_dataset
from tqdm import tqdm


In [ ]:
train_out = base_path + "data/raw/train.txt"
val_out   = base_path + "data/raw/val.txt"


## Cleaning Function

In [ ]:
def clean_text(txt: str):
    if txt is None:
        return None
    txt = txt.strip()
    if len(txt) < 10:
        return None
    return txt


## Load TinyStories in Streaming Mode

In [ ]:
train_stream = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
val_stream   = load_dataset("roneneldan/TinyStories", split="validation", streaming=True)


## Save Cleaned Training Text (Streaming)

In [ ]:
if os.path.exists(train_out):
    print(f"Found existing {train_out}. Skipping download & clean.")
else:
    print(f"Cleaning Training Data...")
    with open(train_out, "w") as f:
        for sample in tqdm(train_stream, desc="Processing Train"):
            txt = clean_text(sample["text"])
            if txt: f.write(txt + "\n")


Found existing /content/drive/MyDrive/Courses/GenAI/SLM from Scratch/data/raw/train.txt. Skipping download & clean.


## Save Cleaned Validation Text (Streaming)

In [ ]:
if os.path.exists(val_out):
    print(f"Found existing {val_out}. Skipping download & clean.")
else:
    print(f"Cleaning Validation Data...")
    with open(val_out, "w") as f:
        for sample in tqdm(val_stream, desc="Processing Val"):
            txt = clean_text(sample["text"])
            if txt: f.write(txt + "\n")


Found existing /content/drive/MyDrive/Courses/GenAI/SLM from Scratch/data/raw/val.txt. Skipping download & clean.


# TOKENIZATION + INPUT/OUTPUT BATCHES

In [ ]:
import os
import numpy as np
from tqdm import tqdm
import tiktoken


In [ ]:
train_bin_path = base_path + "data/tokenized/train.bin"
val_bin_path   = base_path + "data/tokenized/val.bin"


## Load Cleaned Text File Paths

In [ ]:
train_path = base_path+"data/raw/train.txt"
val_path   = base_path+"data/raw/val.txt"

assert os.path.exists(train_path), "train.txt missing"
assert os.path.exists(val_path),   "val.txt missing"


## Initialize GPT-2 Tokenizer

In [ ]:
enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab
vocab_size


50257

## Count Total Tokens (Streaming)

We perform a first pass through the file to count how many total tokens we need to allocate

In [ ]:
def count_tokens_in_file(path):
    total = 0
    with open(path, "r") as f:
        for line in tqdm(f, desc=f"Counting tokens in {path}"):
            total += len(enc.encode(line.strip()))
    return total


## Create Memory-Mapped Files (.bin)

These .bin files store all token IDs, RAM-free.

Second Pass: Tokenize & Write Directly to Disk

In [ ]:
def prepare_bin_file(txt_path, bin_path, desc):
    if os.path.exists(bin_path):
        file_size_bytes = os.path.getsize(bin_path)
        total_tokens = file_size_bytes // 2
        print(f"Found {bin_path}. Loading {total_tokens} tokens directly...")
        data = np.memmap(bin_path, dtype=np.uint16, mode='r', shape=(total_tokens,))
        return data

    print(f"Tokenizing {desc} (First Run)...")
    total_tokens = 0
    with open(txt_path, "r") as f:
        for line in tqdm(f, desc=f"Counting {desc}"):
            total_tokens += len(enc.encode(line.strip()))

    print(f"Allocating {total_tokens} tokens...")
    mmap = np.memmap(bin_path, dtype=np.uint16, mode='w+', shape=(total_tokens,))

    idx = 0
    with open(txt_path, "r") as f:
        for line in tqdm(f, desc=f"Writing {desc}"):
            ids = enc.encode(line.strip())
            mmap[idx : idx + len(ids)] = ids
            idx += len(ids)

    mmap.flush()
    return np.memmap(bin_path, dtype=np.uint16, mode='r', shape=(total_tokens,))

train_data = prepare_bin_file(train_out, train_bin_path, "Train")
val_data   = prepare_bin_file(val_out, val_bin_path, "Val")

print(f"\nFinal Training Tokens: {len(train_data)}")
print(f"Final Validation Tokens: {len(val_data)}")


Found /content/drive/MyDrive/Courses/GenAI/SLM from Scratch/data/tokenized/train.bin. Loading 451644797 tokens directly...
Found /content/drive/MyDrive/Courses/GenAI/SLM from Scratch/data/tokenized/val.bin. Loading 4545541 tokens directly...

Final Training Tokens: 451644797
Final Validation Tokens: 4545541


## Input–Output Batch Generator

In [ ]:
block_size = 512
batch_size = 8
device = "cuda" if torch.cuda.is_available() else "cpu"

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    if device == 'cuda':
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y


## Test the Batch Loader

In [ ]:
xb, yb = get_batch('train')
print(f"Device: {device}")
print(f"Input shape: {xb.shape}")
print(f"Target shape: {yb.shape}")


Device: cpu
Input shape: torch.Size([8, 512])
Target shape: torch.Size([8, 512])


# MODEL ARCHITECTURE (GEMMA-STYLE SLM)

This is the core. We will build a Gemma-style Transformer:

- **RoPE** (Rotary Positional Embeddings): Encodes position mathematically.
- **RMSNorm**: Faster, simpler normalization.
- **MQA** (Multi-Query Attention): Uses 1 Key/Value head for all Query heads (faster inference).
- **Sliding Window Attention**: Saves memory by attending only to recent tokens.
- **GeGLU FeedForward**: A higher-quality activation function.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math


## RMSNorm

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        var = torch.mean(x ** 2, dim=-1, keepdim=True)
        x_norm = x * torch.rsqrt(var + self.eps)
        return self.weight * x_norm


## Rotary Positional Embeddings (RoPE)

In [ ]:
def compute_rope_params(head_dim, theta_base=10000, context_length=4096, device='cpu'):
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    positions = torch.arange(context_length, device=device)
    angles = positions[:, None] * inv_freq[None, :]
    angles = torch.cat([angles, angles], dim=1)
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    return cos, sin


## Apply RoPE

In [ ]:
def apply_rope(x, cos, sin):
    head_dim = x.shape[-1]
    x1 = x[..., :head_dim // 2]
    x2 = x[..., head_dim // 2:]
    cos = cos[:x.shape[1], :].unsqueeze(0).unsqueeze(2)
    sin = sin[:x.shape[1], :].unsqueeze(0).unsqueeze(2)
    rotated = torch.cat((-x2, x1), dim=-1)
    return (x * cos) + (rotated * sin)


## Grouped Query Attention (Supports MQA & Sliding Window)

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.num_heads = cfg["n_heads"]
        self.num_kv_groups = cfg["n_kv_groups"]
        self.head_dim = cfg["head_dim"]
        self.emb_dim = cfg["emb_dim"]
        self.q_dim = self.num_heads * self.head_dim
        self.kv_dim = self.num_kv_groups * self.head_dim
        self.W_q = nn.Linear(self.emb_dim, self.q_dim, bias=False)
        self.W_k = nn.Linear(self.emb_dim, self.kv_dim, bias=False)
        self.W_v = nn.Linear(self.emb_dim, self.kv_dim, bias=False)
        self.out_proj = nn.Linear(self.q_dim, self.emb_dim, bias=False)

    def forward(self, x, cos, sin, mask=None):
        batch, seq_len, _ = x.shape
        q = self.W_q(x).view(batch, seq_len, self.num_heads, self.head_dim)
        k = self.W_k(x).view(batch, seq_len, self.num_kv_groups, self.head_dim)
        v = self.W_v(x).view(batch, seq_len, self.num_kv_groups, self.head_dim)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        groups_per_head = self.num_heads // self.num_kv_groups
        k = k.repeat_interleave(groups_per_head, dim=2)
        v = v.repeat_interleave(groups_per_head, dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        probs = F.softmax(scores, dim=-1)
        output = probs @ v
        output = output.transpose(1, 2).contiguous().view(batch, seq_len, -1)
        return self.out_proj(output)


## Feed-Forward Network with GLU

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False)
        self.w2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False)
        self.w3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], bias=False)

    def forward(self, x):
        return self.w3(F.gelu(self.w1(x)) * self.w2(x))


## Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg["emb_dim"])
        self.attn = GroupedQueryAttention(cfg)
        self.norm2 = RMSNorm(cfg["emb_dim"])
        self.ffn = FeedForward(cfg)

    def forward(self, x, cos, sin, mask):
        x = x + self.attn(self.norm1(x), cos, sin, mask)
        x = x + self.ffn(self.norm2(x))
        return x


## Full Gemma-Style SLM Model

In [ ]:
class Gemma3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.token_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.layers = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.lm_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        cos, sin = compute_rope_params(cfg["head_dim"], context_length=cfg["context_length"])
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def create_mask(self, seq_len, sliding_window=None):
        mask = torch.tril(torch.ones(seq_len, seq_len, device=self.cos.device))
        if sliding_window is not None:
            band_mask = torch.triu(torch.ones(seq_len, seq_len, device=self.cos.device), diagonal=-sliding_window)
            mask = mask * band_mask
        return mask.unsqueeze(0).unsqueeze(0)

    def forward(self, idx, targets=None):
        batch, seq_len = idx.shape
        x = self.token_emb(idx)
        cos = self.cos[:seq_len, :]
        sin = self.sin[:seq_len, :]
        for i, layer in enumerate(self.layers):
            layer_type = self.cfg["layer_types"][i]
            window = self.cfg["sliding_window"] if layer_type == "sliding_attention" else None
            mask = self.create_mask(seq_len, window)
            x = layer(x, cos, sin, mask)
        x = self.final_norm(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None


## Model Configuration

In [ ]:
GEMMA3_CONFIG_270M = {
    "vocab_size": vocab_size,
    "context_length": block_size,
    "emb_dim": 896,
    "n_heads": 8,
    "head_dim": 112,
    "n_layers": 18,
    "hidden_dim": 896 * 4,
    "n_kv_groups": 1,
    "sliding_window": 256,
    "layer_types": [
        "sliding_attention", "sliding_attention", "sliding_attention",
        "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention",
        "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention",
        "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention",
        "full_attention",
        "sliding_attention", "full_attention"
    ]
}


## Instantiate Model

In [ ]:
import gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()


In [ ]:
model = Gemma3Model(GEMMA3_CONFIG_270M)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model Parameters: {total_params / 1e6:.2f} Million")


Model Parameters: 296.02 Million


## TRAINING THE GEMMA-STYLE SLM

We will break this into three parts:
- **Evaluation Helper**: Check model performance without updating weights.
- **Optimizer & Scheduler**: Algorithms that adjust weights.
- **The Main Loop**: The actual training process.

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters=100):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


In [ ]:
import time
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR


In [ ]:
max_iters = 5000
eval_interval = 250
save_interval = 500
learning_rate = 3e-4
warmup_steps = 200
min_lr = 3e-5
grad_accum_steps = 16
batch_size = 8


In [ ]:
param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]

optim_groups = [
    {'params': decay_params, 'weight_decay': 0.1},
    {'params': nodecay_params, 'weight_decay': 0.0}
]

optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95))


In [ ]:
scheduler_warmup = LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
scheduler_decay = CosineAnnealingLR(optimizer, T_max=max_iters - warmup_steps, eta_min=min_lr)
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps])

print("Optimizer and Scheduler ready!")


Optimizer and Scheduler ready!


In [ ]:
print("Compiling model for faster training on T4...")
model = torch.compile(model)


## TRAINING LOOP

Includes Mixed Precision (torch.amp) — calculates some operations in float16 (faster, less memory) while keeping weights in float32.

In [ ]:
from contextlib import nullcontext

dtype = 'float16'
ptdtype = torch.float16

scaler = torch.amp.GradScaler('cuda', enabled=True)
ctx = torch.amp.autocast(device_type='cuda', dtype=ptdtype)

print(f"Starting training with {dtype} precision...")
print(f"Effective Batch Size: {batch_size * grad_accum_steps}")

checkpoint_path = base_path + "model/checkpoints/best_model.pt"
start_iter = 0
t0 = time.time()
best_val_loss = float('inf')
train_losses = []
val_losses = []

if os.path.exists(checkpoint_path):
    print(f"Checkpoint found at {checkpoint_path}")
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    initial_losses = estimate_loss(model)
    best_val_loss = initial_losses['val']
    print(f"Resumed! Previous Best Val Loss: {best_val_loss:.4f}")
else:
    print("No checkpoint found. Starting from scratch.")

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        dt = time.time() - t0
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, time {dt:.2f}s")
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), base_path + "model/checkpoints/best_model.pt")
            print(f"Saved new best model (Loss: {best_val_loss:.4f})")
        t0 = time.time()

    for micro_step in range(grad_accum_steps):
        X, Y = get_batch('train')
        if X.shape[0] != batch_size: X, Y = X[:batch_size], Y[:batch_size]
        with ctx:
            logits, loss = model(X, Y)
            loss = loss / grad_accum_steps
        scaler.scale(loss).backward()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    scheduler.step()

print("Training Complete! Best model saved.")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Evaluation Steps")
plt.ylabel("Loss")
plt.title("SLM Training Progress")
plt.legend()
plt.grid(True)
plt.show()


# INFERENCE + SAMPLING + DEPLOYMENT

Switch model from Learning Mode to Generation Mode.

In [ ]:
import torch
import torch.nn.functional as F
import tiktoken
from tqdm import tqdm


## Load the Trained Model

In [ ]:
print("Initializing model structure...")
model = Gemma3Model(GEMMA3_CONFIG_270M)
model.to(device)

checkpoint_path = base_path + "model/checkpoints/best_model.pt"

if os.path.exists(checkpoint_path):
    print(f"Loading weights from {checkpoint_path}...")
    state_dict = torch.load(checkpoint_path, map_location=device)
    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    model.load_state_dict(state_dict)
    print("Model loaded successfully!")
else:
    print("No checkpoint found! Using random weights.")

model.eval()


Initializing model structure...
Loading weights from /content/drive/MyDrive/Courses/GenAI/SLM from Scratch/model/checkpoints/best_model.pt...
Model loaded successfully!


## Define the Generator Function

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=200, temperature=0.8, top_k=50):
    input_ids = enc.encode(prompt)
    idx = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]
        logits = logits / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)

    output = idx[0].tolist()
    return enc.decode(output)


## Test Generation

In [ ]:
print("-" * 50)
prompt1 = "Once upon a time, there was a little girl named Lily."
print(generate(prompt1, max_new_tokens=150, temperature=0.7))

print("\n" + "-" * 50)
prompt2 = "One day, a big red dragon flew"
print(generate(prompt2, max_new_tokens=150, temperature=0.9))

print("\n" + "-" * 50)
prompt3 = "The cat sat on the mat and"
print(generate(prompt3, max_new_tokens=100, temperature=0.4))
